In [2]:
from torch import nn
import torch

from Reconstruction import SpatialBroadcastDecoder
from SlotModel import CompetitiveSlotAttention, PromptSlotInitializer 
# core PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# CLIP (text + image encoders)
from transformers import CLIPModel, CLIPProcessor

# FG-ARI metric
from sklearn.metrics import adjusted_rand_score
import numpy as np

# training utilities
import os
import time
import glob

# visualization (only needed for the "inspect one batch" step, not training itself)
import matplotlib.pyplot as plt


c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0828 13:22:28.383000 44808 torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0828 13:22:28.531000 44808 torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:

def build_labels(batch):
    B = batch["visibility"].shape[0]
    labels_per_sample = []
    for i in range(B):
        vis = batch["visibility"][i]          # [11], 1 = real object, entity 0 = background
        labels = []
        for obj_idx in range(1, 11):           # skip index 0, that's background
            if vis[obj_idx] == 1:
                label = f"{batch['size'][i][obj_idx]} {batch['color'][i][obj_idx]} " \
                        f"{batch['material'][i][obj_idx]} {batch['shape'][i][obj_idx]}"
                labels.append(label)
        labels_per_sample.append(labels)
    return labels_per_sample

def lr_at_step(step, cfg):
    if step < cfg.warmup_steps:
        warmup_factor = step / cfg.warmup_steps
    else:
        warmup_factor = 1.0
    decay_factor = cfg.decay_rate ** (step / cfg.decay_steps)
    return cfg.base_lr * warmup_factor * decay_factor
def compute_fg_ari(masks, label_map):
    pred_labels = masks.squeeze(2).argmax(dim=1)  # which slot "won" each pixel
    scores = []
    for b in range(masks.shape[0]):
        gt = label_map[b].cpu().numpy().reshape(-1)
        pred = pred_labels[b].cpu().numpy().reshape(-1)
        fg = gt != 0
        scores.append(adjusted_rand_score(gt[fg], pred[fg]))
    return sum(scores) / len(scores)
@torch.no_grad()
def evaluate(model, val_loader, device, max_batches=20):
    model.eval()
    mse_total, ari_total, n = 0.0, 0.0, 0
    for i, batch in enumerate(val_loader):
        if i >= max_batches:
            break
        recon, masks, _ = model(batch)
        mse_total += F.mse_loss(recon, batch["image"].to(device)).item()
        ari_total += compute_fg_ari(masks, batch["label_map"].to(device))
        n += 1
    model.train()
    return mse_total / n, ari_total / n